# Criteo Uplift Analysis

This notebook analyzes a large randomized advertising experiment from Criteo (about 14 million users, roughly 85% treated). It covers a data audit, A/B test analysis (average treatment effect, confidence intervals, power, intent-to-treat vs. exposure effects), uplift modeling evaluated with Qini curves, and a targeting decision that estimates the value of showing ads only to the users most likely to respond.

## 0. Setup

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import duckdb
import sklearn
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEED = 42
np.random.seed(SEED)

print("pandas   ", pd.__version__)
print("numpy    ", np.__version__)
print("duckdb   ", duckdb.__version__)
print("sklearn  ", sklearn.__version__)
print("lightgbm ", lgb.__version__)

pandas    2.3.3
numpy     2.1.3
duckdb    1.3.2
sklearn   1.6.1
lightgbm  4.6.0


### 0.1 Locate input files

In [2]:
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{path}  ({size_mb:,.1f} MB)")

/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv  (3,097.6 MB)
/kaggle/input/datasets/guilinzhao/criteo-uplift-parquet/criteo_uplift.parquet  (104.5 MB)


### 0.2 First look at the raw file

In [3]:
csv_files = sorted(glob.glob("/kaggle/input/**/*.csv*", recursive=True))
print(csv_files)

RAW_PATH = csv_files[0]
print("Using:", RAW_PATH)

sample = pd.read_csv(RAW_PATH, nrows=5)
print(sample.shape)
print(sample.dtypes)
sample

['/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv']
Using: /kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv
(5, 16)
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0


### 0.3 Convert CSV to Parquet (run once)

The raw CSV is about 3.1 GB. It is converted once to a compressed Parquet file with compact types (float32 features, int8 labels) and saved as a Kaggle dataset. Later runs set `RUN_DATA_PREP = False` and read the saved file directly.

In [4]:
import time

RUN_DATA_PREP = False  # Set to False after the Parquet file is saved as a Kaggle dataset

PARQUET_NAME = "criteo_uplift.parquet"
WORK_PATH = f"/kaggle/working/{PARQUET_NAME}"

FEATURE_COLS = [f"f{i}" for i in range(12)]
LABEL_COLS = ["treatment", "conversion", "visit", "exposure"]

if RUN_DATA_PREP:
    start = time.time()
    con = duckdb.connect()

    select_sql = ",\n        ".join(
        [f"CAST({c} AS FLOAT) AS {c}" for c in FEATURE_COLS]
        + [f"CAST({c} AS TINYINT) AS {c}" for c in LABEL_COLS]
    )

    con.execute(f"""
        COPY (
            SELECT
        {select_sql}
            FROM read_csv('{RAW_PATH}', header = true)
        ) TO '{WORK_PATH}' (FORMAT parquet, COMPRESSION zstd)
    """)

    raw_rows = con.execute(
        f"SELECT COUNT(*) FROM read_csv('{RAW_PATH}', header = true)"
    ).fetchall()[0][0]
    pq_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{WORK_PATH}')"
    ).fetchall()[0][0]
    con.close()

    print(f"Raw CSV rows:     {raw_rows:,}")
    print(f"Parquet rows:     {pq_rows:,}")
    print(f"Row counts match: {raw_rows == pq_rows}")
    print(f"Parquet size:     {os.path.getsize(WORK_PATH) / 1024**2:,.1f} MB")
    print(f"Elapsed:          {time.time() - start:,.1f} s")
else:
    print("Skipping data prep; the saved Parquet will be loaded below.")

Skipping data prep; the saved Parquet will be loaded below.


### 0.4 Load the prepared data

In [5]:
if RUN_DATA_PREP:
    DATA_PATH = WORK_PATH
else:
    found = glob.glob(f"/kaggle/input/**/{PARQUET_NAME}", recursive=True)
    assert found, "Prepared Parquet not found. Attach the prepared dataset as an input."
    DATA_PATH = found[0]

df = pd.read_parquet(DATA_PATH)

print("Loaded from:", DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print(df.dtypes)
df.head()

Loaded from: /kaggle/input/datasets/guilinzhao/criteo-uplift-parquet/criteo_uplift.parquet
Shape: 13,979,592 rows x 16 columns
Memory: 693.3 MB
f0            float32
f1            float32
f2            float32
f3            float32
f4            float32
f5            float32
f6            float32
f7            float32
f8            float32
f9            float32
f10           float32
f11           float32
treatment        int8
conversion       int8
visit            int8
exposure         int8
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
